# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one pseudonymized content item for one pseudonymized client on one report date in the `fact_content_daily_performance` table.

**Time window:** For development and verification, I will use the mid-panel month `2026-03` rather than the final `_sample` month. The decision features will use information available before the outcome window, so future outcome information is not used at decision time.


In [1]:
# This cell is for CODE
# Section 1 — Verify the unit of analysis and time window

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

month = "2026-03"

print("Warehouse connection configured.")
print("Development month:", month)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Warehouse connection configured.
Development month: 2026-03


In [4]:
# Section 1 — Grain verification

grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
"""

grain_check = con.sql(grain_query).df()

display(grain_check)

,total_rows,distinct_dates,distinct_clients,distinct_content_items
0,9841378,31,55,331437


In [3]:
# Diagnostic check — inspect the March 2026 partition directly

test_query = f"""
SELECT
    month,
    COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY month
"""

display(con.sql(test_query).df())

,month,rows
0,2026-03,9841378


In [5]:
# Section 1 — Verify that the declared grain is unique

duplicate_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
"""

duplicate_check = con.sql(duplicate_query).df()

display(duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,duplicate_rows
0,9841378,0


## 2. Fields: feature / label / context / excluded

### Feature

* `gsc_impressions` — recent search visibility.
* `gsc_clicks` — recent search traffic from Google.
* `gsc_avg_position` — observed average search position.
* `gsc_data_available` — whether GSC data is available for the row.
* `report_date` — used to establish the decision date and time ordering.

### Label

* A future-period change in search performance, derived from later observations after the decision date. This is the outcome used to assess whether a page should be prioritized for refresh.

### Context

* `client_hash_id` — identifies the pseudonymized client.
* `content_hash_id` — identifies the pseudonymized content item.
* `client_has_gsc` — indicates whether the client has GSC access.
* `client_has_ga4` — indicates whether the client has GA4 access.

### Excluded

* `gsc_clicks`, `gsc_impressions`, and `gsc_avg_position` are used only as current-period features, not as future-derived values.
* GA4 fields (`ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, and session-source fields) are excluded from the first feature frame because this lane is focused on search-performance signals and not all rows have GA4 availability.
* `month` is used for partition filtering rather than as a modeling feature.
* `client_hash_id` and `content_hash_id` are identifiers, not predictive signals.


In [6]:
# This cell is for CODE
# Section 2 — Inspect available fields

columns_query = f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
"""

columns = con.sql(columns_query).df()

display(columns)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 — Grain verification

For the March 2026 development slice, I verify that the declared grain is one row per `report_date`, `client_hash_id`, and `content_hash_id`. A duplicate count of zero supports this claim.


### Query 2 — Row count and date span

I verify the size and date coverage of the March 2026 slice so the development window is measured rather than assumed.


### Query 3 — GSC availability

Because the lane uses Google Search Console signals, I measure how many March 2026 rows have GSC data available. The filter deliberately uses `IS TRUE`, so only rows explicitly marked as available survive.


### Five-feature frame — Content Refresh Prioritization

I use five signals from the March 2026 development slice. Each is treated as information available at the decision moment rather than as a future outcome.

| Feature              | Why it is knowable at the decision moment                                                        |
| -------------------- | ------------------------------------------------------------------------------------------------ |
| `gsc_impressions`    | It is an observed search-visibility measure for the current reporting period.                    |
| `gsc_clicks`         | It is an observed search-click measure for the current reporting period.                         |
| `gsc_avg_position`   | It is an observed search-position measure for the current reporting period.                      |
| `gsc_data_available` | It records whether GSC data is available for the current row.                                    |
| `ga4_pageviews`      | It is an observed pageview measure for the current reporting period, when GA4 data is available. |

These features are intended for decision support and prioritization. They are not treated as causal predictors of future performance.


### Deliberate leakage experiment

To demonstrate label leakage, I define a future outcome using April 2026 performance while treating March 2026 as the decision period. The honest features contain only March information. For the leakage demonstration, I intentionally add the future April impression value as a feature. Because this value is only known after the decision moment, a model using it would have access to the outcome window and would produce an artificially optimistic result.


In [7]:
# This cell is for CODE
# Query 1 — Verify the declared grain

grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
"""

display(con.sql(grain_query).df())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,duplicate_rows
0,9841378,0


In [8]:
# Query 2 — Row count and date span

count_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
"""

display(con.sql(count_window_query).df())

,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# Query 3 — Verify GSC availability using IS TRUE

availability_query = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
  AND gsc_data_available IS TRUE
"""

display(con.sql(availability_query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


In [10]:
# Five-feature frame for Content Refresh Prioritization

feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_data_available,
    ga4_pageviews
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '{month}'
  AND gsc_data_available IS TRUE
LIMIT 1000
"""

features = con.sql(feature_query).df()

display(features.head())
print("Feature frame shape:", features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,ga4_pageviews
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,True,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,True,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,True,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,True,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,True,<NA>


Feature frame shape: (1000, 8)


In [11]:
# Create March decision features and April future outcome

march_april_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_impressions,
    CASE
        WHEN a.april_impressions > m.march_impressions
        THEN 1
        ELSE 0
    END AS label
FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
"""

march_april = con.sql(march_april_query).df()

display(march_april.head())
print("Rows with March + April data:", len(march_april))
print("Positive outcomes:", march_april["label"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,april_impressions,label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,6787.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,405.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,8475.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,6091.0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,287.0,0


Rows with March + April data: 158549
Positive outcomes: 61622


### Leakage score comparison

I compare a simple baseline using only information available at the March decision moment with a deliberately leaked version that includes `april_impressions`. The April value is future information and is therefore unavailable when the refresh-prioritization decision is made. A substantially stronger leaked score demonstrates why future outcome fields must be removed from the feature set.


In [12]:
# Deliberate leakage demonstration

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Keep only rows with a valid future outcome
leak_df = march_april.dropna(
    subset=[
        "march_impressions",
        "march_clicks",
        "march_avg_position",
        "april_impressions",
        "label"
    ]
).copy()

# Honest features: information available at decision time
X_honest = leak_df[
    [
        "march_impressions",
        "march_clicks",
        "march_avg_position"
    ]
]

# Leaky features: intentionally includes future April information
X_leaky = leak_df[
    [
        "march_impressions",
        "march_clicks",
        "march_avg_position",
        "april_impressions"
    ]
]

y = leak_df["label"]

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_honest,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

leaky_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

honest_model.fit(Xh_train, yh_train)
leaky_model.fit(Xl_train, yl_train)

honest_score = accuracy_score(
    yh_test,
    honest_model.predict(Xh_test)
)

leaky_score = accuracy_score(
    yl_test,
    leaky_model.predict(Xl_test)
)

print(f"Honest feature score: {honest_score:.3f}")
print(f"Leaky feature score:  {leaky_score:.3f}")
print(f"Leakage improvement:   {leaky_score - honest_score:+.3f}")

Honest feature score: 0.621
Leaky feature score:  0.658
Leakage improvement:   +0.038


### Leakage removed

The leakage experiment increased the quick-test accuracy from 0.621 to 0.658 after adding `april_impressions`, a future-period value. Although the improvement was only 0.038 in this split, the feature is invalid because April performance is not knowable at the March decision moment.

I therefore remove `april_impressions` and retain only information available at the decision moment. The final feature set is limited to current-period search-performance and availability signals. The leakage experiment is retained as a demonstration, but the leaked field is not used for the final feature frame or decision-support logic.


In [13]:
# Final honest feature set — future information removed

honest_features = leak_df[
    [
        "march_impressions",
        "march_clicks",
        "march_avg_position"
    ]
].copy()

print("Final honest features:")
print(list(honest_features.columns))
print("\nFuture-derived feature removed: april_impressions")

Final honest features:
['march_impressions', 'march_clicks', 'march_avg_position']

Future-derived feature removed: april_impressions


## 4. Data limits

### Data limits

This dataset supports observed and directional decision support, but it cannot establish that refreshing a page causes better search performance.

The history is unbalanced across clients and content items, so not every item has the same amount of historical data available. GSC availability also varies, meaning search-performance signals are not equally observable for every row. In addition, rolling or fixed performance windows can overlap across decision periods, so nearby observations are not fully independent.

For this reason, results should be interpreted as measured associations for prioritization rather than causal effects or guaranteed outcomes.


**Named limitation:** The March 2026 slice has incomplete and uneven data availability across clients and content items, so results from this slice should not be assumed to represent the entire warehouse population.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.